# 03 — Evaluación del sistema GraphRAG (metodología RAGAS)

Este notebook evalúa el pipeline usando las tres métricas **RAGAS** descritas en
clase. El enfoque **no** requiere la librería RAGAS —
todas las métricas se implementan como llamadas estructuradas al LLM.

### Las tres métricas

| Métrica | Qué mide | Cómo |
|---|---|---|
| **context_recall** | ¿El recuperador obtuvo chunks que cubren la verdad esperada? | Cada frase del ground truth se verifica: ¿puede atribuirse al contexto recuperado? |
| **faithfulness** | ¿La respuesta contiene solo cosas respaldadas por el contexto? | La respuesta se descompone en afirmaciones atómicas; cada una se verifica contra el contexto. |
| **answer_correctness** | ¿Cuánto se superpone la respuesta con el ground truth? | Las afirmaciones se clasifican en TP / FP / FN; se calcula el F1. |

### El pipeline de evaluación

```
CSV de benchmark
  (question ; cypher)
	  │
	  ▼
  load_dataset()       ← lee el CSV, ejecuta Cypher para obtener ground truths dinámicos
	  │
	  ▼
  run_benchmark()      ← llama al agente RAG para cada pregunta, registra la latencia
	  │
	  ▼
  evaluate_results()   ← puntúa cada fila con las tres métricas RAGAS
	  │
	  ▼
  print_summary()      ← tabla agregada
```

**Requisito previo:** Ejecutar `01_ingestion_demo.ipynb` para cargar los datos de Einstein en Neo4j.

## 1. Configuración

In [16]:
import pandas as pd
from graph.neo4j_manager import Neo4jManager
from agents.multi_agent_system import MultiAgentSystem
from evaluation.evaluator import RAGEvaluator
import matplotlib.pyplot as plt

pd.set_option('display.max_colwidth', 60)
pd.set_option('display.float_format', '{:.3f}'.format)


In [17]:
neo4j = Neo4jManager()
rag = MultiAgentSystem(neo4j, verbose=False)
evaluator = RAGEvaluator(rag, neo4j)
print("Sistema listo.")


Sistema listo.


## 2. Inspección de métricas — pregunta a pregunta

Antes de ejecutar el benchmark completo, es instructivo ver qué hace cada métrica
internamente. Tomamos una sola pregunta y llamamos a cada métrica de forma manual.

In [ ]:
question     = "Where do the ogres from the 'Momotaro' folktale live?"
ground_truth = "The ogres from the Momotaro folktale live in a fortified castle, known as the Ogres' Island castle."

# Preguntamos al agente
answer, context = evaluator.get_answer(question)
print(f"\nRespuesta: {answer}")
print(f"Chunks: {context}")
print(f"Chunks recuperados: {len(context)}")

In [ ]:
# --- context_recall ---
# Objetivo: ¿puede atribuirse cada frase del ground truth al contexto recuperado?
# Puntuación = (frases atribuidas) / (total de frases)  →  rango [0, 1]
recall_result = evaluator.evaluate_context_recall(question, ground_truth, context, verbose=True)

print("context_recall")
print(f"  puntuación : {recall_result.recall:.3f}")
print(f"  frases     : {recall_result.sentences}")
print(f"  atribuidas : {recall_result.attributions}")
print(f"  razonamiento: {recall_result.reasoning}")

In [ ]:
# --- faithfulness ---
# Paso 1: descomponer la respuesta en afirmaciones atómicas sin pronombres.
# Paso 2: verificar si cada afirmación puede inferirse del contexto.
# Puntuación = (afirmaciones respaldadas) / (total)  →  rango [0, 1]
faith_result = evaluator.evaluate_faithfulness(question, str(answer), context, verbose=True)

print("faithfulness")
print(f"  puntuación  : {faith_result.faithfulness:.3f}")
print(f"  afirmaciones: {faith_result.statements}")
print(f"  veredictos  : {faith_result.verdicts}")

In [ ]:
# --- answer_correctness ---
# Se descomponen tanto la respuesta como el ground truth en afirmaciones.
# Cada afirmación se clasifica en TP / FP / FN.
# Puntuación = F1(precisión, recall) calculado a partir de los conteos TP / FP / FN.
corr_result = evaluator.evaluate_answer_correctness(question, str(answer), ground_truth, verbose=True)

print("answer_correctness")
print(f"  F1        : {corr_result.answer_correctness:.3f}")
print(f"  precisión : {corr_result.precision:.3f}")
print(f"  recall    : {corr_result.recall:.3f}")
print(f"  TP={corr_result.tp}  FP={corr_result.fp}  FN={corr_result.fn}")


## 3. Pipeline de benchmark completo

### Formato del dataset de benchmark

El benchmark es un CSV delimitado por punto y coma con dos columnas:
- `question` — la pregunta en lenguaje natural
- `cypher` — una consulta Cypher que se **ejecuta en tiempo de ejecución** para obtener el ground truth

Este diseño garantiza que los ground truths estén siempre sincronizados con el estado
real del grafo: no hay cadenas de texto codificadas que queden desactualizadas al cambiar los datos.

### Categorías de preguntas

| Categoría | Ejemplos | Propósito |
|---|---|---|
| **Saludos** | "Hello", "What can you do?" | Evaluar respuesta conversacional |
| **Fuera de ámbito** | "Weather?", "World Cup?" | Evaluar rechazo correcto |
| **Datos ausentes** | "Einstein's middle name?" | Evaluar honestidad del sistema |
| **Hechos simples** | "Where was Einstein born?" | Evaluar recuperación directa |
| **Consultas relacionales** | "Which institutions did Einstein work at?" | Evaluar traversal de grafo |
| **Consultas complejas** | "Where did Einstein live after Germany?" | Evaluar razonamiento multi-salto |
| **Agregaciones** | "How many Person nodes?" | Evaluar text2cypher |


In [ ]:
dataset = evaluator.load_dataset('./data/benchmark_data.csv')
print(f"Benchmark CSV: {len(dataset)} preguntas")
dataset.head()


Benchmark CSV: 36 preguntas


,question,cypher,type
0,Hello,"RETURN ""Hello! I'm a knowledge assistant focused on folk...",greeting
1,What can you do?,"RETURN ""I can answer questions about folktales: their na...",greeting
2,What is the weather like today?,"RETURN ""This question is outside my scope."" AS ground_truth",out_of_scope
3,Who won the last football World Cup?,"RETURN ""This question is outside my scope."" AS ground_truth",out_of_scope
4,What is the capital of France?,RETURN 'This question is outside my scope.' AS ground_truth,out_of_scope


In [ ]:
for i, row in dataset.iterrows():
	question = row["question"]
	cypher = row["cypher"]
	
	result = neo4j.execute_query(cypher)
	print(f"Question {i}: {result}")


Question 0: [{'ground_truth': "Hello! I'm a knowledge assistant focused on folktales from all around the world. You can ask me about their characters, relationships, themes or how they are structured."}]
Question 1: [{'ground_truth': 'I can answer questions about folktales: their narrative structure, the characters that make them up and their features, the relevant places and objects that appear, among others.'}]
Question 2: [{'ground_truth': 'This question is outside my scope.'}]
Question 3: [{'ground_truth': 'This question is outside my scope.'}]
Question 4: [{'ground_truth': 'This question is outside my scope.'}]
Question 5: [{'ground_truth': 'This question is outside my scope.'}]
Question 6: [{'ground_truth': 'This information is not in the knowledge base.'}]
Question 7: [{'ground_truth': 'This information is not in the knowledge base.'}]
Question 8: [{'ground_truth': 'This information is not in the knowledge base.'}]
Question 9: [{'ground_truth': 'This information is not in the kn

In [ ]:
counts = dataset["type"].value_counts()

plt.figure(figsize=(8,5))
counts.plot(kind="bar")
plt.title("Distribución de tipos de preguntas")
plt.xlabel("Tipo")
plt.ylabel("Cantidad")
plt.xticks(rotation=45)
plt.tight_layout()

plt.show()


### 3.1 `run_benchmark()` — ejecutar ground truths Cypher y llamar al agente

Para cada fila:
1. Ejecuta la consulta Cypher → string con el ground truth
2. Llama a `rag.answer(question)` → respuesta del agente + contextos recuperados
3. Registra la latencia en segundos

In [ ]:
results_df = evaluator.run_benchmark(dataset, verbose=True)

print(f"\nBenchmark completo — {len(results_df)} filas")
results_df[['question', 'ground_truth', 'answer', 'latency']]

### 3.2 `evaluate_results()` — puntuar cada fila con las tres métricas RAGAS

Este paso es el más intensivo en LLM: cada fila hace ~5 llamadas al LLM
(context_recall × 1, faithfulness × 2, answer_correctness × 3).
Esperar unos 10–30 segundos por fila dependiendo del modelo.

In [ ]:
scored_df = evaluator.evaluate_results(results_df, verbose=True)

print(f"\nPuntuación completada.")
scored_df[['question', 'context_recall', 'faithfulness', 'answer_correctness', 'latency']]

### 3.3 `print_summary()` — resultados agregados

El resumen reproduce la tabla de resumen de benchmark que hemos comentado en clase.
Un sistema bien ajustado debería superar 0.7 en las tres métricas.

In [ ]:
evaluator.print_summary(scored_df)

## 4. Análisis de resultados

### 4.1 Distribución por métrica

In [ ]:
metrics = ['context_recall', 'faithfulness', 'answer_correctness']
colors  = ['steelblue', 'seagreen', 'darkorange']

fig, axes = plt.subplots(1, 3, figsize=(13, 4))
fig.suptitle('Distribución de métricas RAGAS', fontsize=13)

for ax, metric, color in zip(axes, metrics, colors):
	ax.hist(scored_df[metric].dropna(), bins=5, color=color, edgecolor='white', alpha=0.85)
	ax.axvline(scored_df[metric].mean(), color='black', linestyle='--', linewidth=1.2,
			   label=f'media={scored_df[metric].mean():.2f}')
	ax.set_title(metric)
	ax.set_xlabel('Puntuación')
	ax.set_xlim(0, 1)
	ax.legend(fontsize=8)

plt.tight_layout()
plt.show()

In [ ]:
# Distribución de latencia
fig, ax = plt.subplots(figsize=(6, 3))
ax.barh(range(len(scored_df)), scored_df['latency'], color='slategray')
ax.set_yticks(range(len(scored_df)))
ax.set_yticklabels(
	[q[:40] + '...' if len(q) > 40 else q for q in scored_df['question']],
	fontsize=8
)
ax.set_xlabel('Latencia (s)')
ax.set_title('Latencia de respuesta por pregunta')
plt.tight_layout()
plt.show()

### 4.2 Mejor y peor caso por answer_correctness

Inspeccionar los extremos ayuda a diagnosticar *por qué* falla el sistema:
- `context_recall` bajo → el recuperador no encontró los chunks relevantes → ajustar `chunk_size` o `top_k`
- `faithfulness` bajo → el LLM alucinó → mejorar el prompt de respuesta o añadir instrucciones de anclaje
- `answer_correctness` bajo con `faithfulness` alto → el contexto recuperado era incompleto

In [ ]:
def show_case(label, row):
	print(f"{'='*70}")
	print(label)
	print(f"{'='*70}")
	print(f"Question        : {row['question']}")
	print(f"Ground truth    : {str(row['ground_truth'])[:200]}")
	print(f"Answer          : {str(row['answer'])[:200]}")
	print(f"answer_correctness : {row['answer_correctness']:.3f}")
	print(f"context_recall     : {row['context_recall']:.3f}")
	print(f"faithfulness       : {row['faithfulness']:.3f}")
	print()

best_idx  = scored_df['answer_correctness'].idxmax()
worst_idx = scored_df['answer_correctness'].idxmin()

show_case("MEJOR CASO",  scored_df.loc[best_idx])
show_case("PEOR CASO", scored_df.loc[worst_idx])

### 4.3 Correlación entre métricas

¿Son las tres métricas independientes o tienden a moverse juntas?
Una correlación fuerte entre `context_recall` y `answer_correctness` confirmaría
que la calidad de la recuperación es el principal cuello de botella.

In [ ]:
corr = scored_df[metrics].corr()
print("Correlación de Pearson entre métricas:")
print(corr.round(3))

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(10, 4))

axes[0].scatter(scored_df['context_recall'], scored_df['answer_correctness'],
				color='steelblue', edgecolors='white', s=80, alpha=0.85)
axes[0].set_xlabel('context_recall')
axes[0].set_ylabel('answer_correctness')
axes[0].set_title('Recall de contexto vs Corrección de respuesta')
axes[0].set_xlim(0, 1); axes[0].set_ylim(0, 1)

axes[1].scatter(scored_df['faithfulness'], scored_df['answer_correctness'],
				color='seagreen', edgecolors='white', s=80, alpha=0.85)
axes[1].set_xlabel('faithfulness')
axes[1].set_ylabel('answer_correctness')
axes[1].set_title('Fidelidad vs Corrección de respuesta')
axes[1].set_xlim(0, 1); axes[1].set_ylim(0, 1)

plt.suptitle('Correlaciones entre métricas', fontsize=12)
plt.tight_layout()
plt.show()

## 5. Guardar resultados

In [ ]:
from datetime import datetime

timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")
out_path = f"./data/evaluation_results_{timestamp}.csv"
scored_df.to_csv(out_path, index=False)
print(f"Resultados guardados en {out_path}")


In [ ]:
neo4j.close()
